# anvesa: distil jina-v2-code (WQL-path cards) into a small modern encoder

Runtime → Change runtime type → **T4 GPU**. Make the bundle on your machine first:

```
python tooling/poc/distill/export_cards.py . <work>
python tooling/poc/distill/make_pairs.py <work> <commerce-wql index.db>
python tooling/poc/distill/bundle.py <work>      # -> <work>/distill_bundle.zip
```

Stages, each resumable (they write to `WORK`, which can be on Drive):
1. **queries**: Qwen2.5-Coder-3B writes a search query per card (≈20–30 min for 20k cards)
2. **teacher**: jina-v2-code embeds everything, mines hard negatives, records its scores (≈5–10 min)
3. **train**: each student (≈10–40 min each)
4. **export**: ONNX fp32 for anvesa, zipped for download

Do **not** quantize a ModernBERT/Ettin student to int8: dynamic int8 took a trained one from 0.40 to 0.02 MRR in the local test.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install "transformers>=4.57,<5" "sentence-transformers>=5.1,<6" peft onnx onnxruntime

In [ ]:
USE_DRIVE = True   # keep WORK on Drive so a disconnect does not lose the teacher vectors or runs
import os, zipfile, pathlib
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = pathlib.Path('/content/drive/MyDrive/anvesa-distill')
else:
    BASE = pathlib.Path('/content')
BASE.mkdir(parents=True, exist_ok=True)
bundle = BASE / 'distill_bundle.zip'
if not bundle.exists():
    from google.colab import files
    up = files.upload()  # choose distill_bundle.zip
    bundle.write_bytes(up['distill_bundle.zip'])
with zipfile.ZipFile(bundle) as z:
    z.extractall(BASE)
WORK = BASE / 'distill'
os.chdir(WORK)
!ls -la data | head

In [ ]:
# 1. LLM-written queries. Doc-derived pairs alone are few and read like docs, not like what people type.
#    Locally the 1.5B generator wrote some vague or wrong queries; the 3B one should do better (not yet measured).
#    The teacher stage then drops any query whose card the teacher cannot find in its top 10 (--llm-top).
N_QUERIES = 20000
GENERATOR = 'Qwen/Qwen2.5-Coder-3B-Instruct'
if not (WORK / 'data/pairs_llm.jsonl').exists():
    !python train.py queries {WORK} --n {N_QUERIES} --gen-batch 64 --generator {GENERATOR}
!wc -l data/pairs.jsonl data/pairs_llm.jsonl

In [ ]:
# 2. Teacher: vectors are cached in out/, so rerunning with another --margin, --negatives or --llm-top is quick.
!python train.py teacher {WORK} --negatives 7 --margin 0.02 --llm-top 10

In [ ]:
# 3. Students. Each run keeps its best epoch on the val repos (mean of val_doc and val_llm), never on commerce.
#    First run (ettin-17m, --kl 1, 3 epochs): commerce 0.634, below MiniLM's 0.649; val_doc peaked at epoch 2,
#    val_llm stayed at 0.47. So: the bigger student first, and 4x the weight on matching the teacher's ranking.
RUNS = [
    ('ettin-32m-kl4',        '--student jhu-clsp/ettin-encoder-32m --epochs 4 --lr 8e-5 --kl 4'),
    ('ettin-17m-kl4',        '--student jhu-clsp/ettin-encoder-17m --epochs 4 --lr 1e-4 --kl 4'),
    ('ettin-32m-kl4-lora16', '--student jhu-clsp/ettin-encoder-32m --epochs 4 --lr 5e-4 --kl 4 --lora 16'),
]
DONE = ['ettin-17m']  # earlier runs, shown in the table below for comparison (ettin-17m: --kl 1, 3 epochs)
for name, flags in RUNS:
    if (WORK / 'out' / name / 'history.json').exists():
        print('done already:', name); continue
    !python train.py train {WORK} --name {name} {flags} --batch 64 --negatives 7

In [ ]:
import json
BASELINES = {'all-MiniLM-L6-v2 (anvesa today)': 0.649, 'gte-small + WQL path': 0.700, 'jina-v2-code + WQL path (teacher)': 0.775}
print(f"{'run':28} {'commerce':>9} {'val doc':>8} {'val llm':>8} {'epoch':>6}")
for name, value in BASELINES.items():
    print(f'{name:28} {value:9.3f}   (measured locally, anvesa ONNX)')
for name in DONE + [n for n, _ in RUNS]:
    h = WORK / 'out' / name / 'history.json'
    if h.exists():
        hist = json.loads(h.read_text())
        kept = next((e for e in hist if e.get('selected')), hist[-1])  # runs before selection kept the last epoch
        print(f"{name:28} {kept['commerce']:9.3f} {kept.get('val_doc', float('nan')):8.3f} "
              f"{kept.get('val_llm', float('nan')):8.3f} {kept['epoch']:6}")

In [ ]:
# 4. Export to ONNX (fp32) and download. Install on your machine with:
#    anvesa model install <name> --from <unzipped>/<name> --pooling mean
import shutil
pack = pathlib.Path('/content/students')
pack.mkdir(exist_ok=True)
for name, _ in RUNS:
    if not (WORK / 'out' / name / 'history.json').exists(): continue
    if not (WORK / 'out' / name / 'onnx' / 'model.onnx').exists():
        !python train.py export {WORK} --run {name}
    dst = pack / name
    dst.mkdir(exist_ok=True)
    for f in ('model.onnx', 'tokenizer.json', 'meta.json'):
        shutil.copy(WORK / 'out' / name / 'onnx' / f, dst / f)
    shutil.copy(WORK / 'out' / name / 'history.json', dst / 'history.json')
shutil.make_archive('/content/students', 'zip', pack)
from google.colab import files
files.download('/content/students.zip')